In [9]:
import nltk
import numpy as np
import spacy
import string
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.metrics.distance import edit_distance

In [10]:
print("Downloading NLTK resources...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("Loading spaCy English model...")
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model loaded successfully.\n")
except OSError:
    print("Error: spaCy model not found. Please run 'python -m spacy download en_core_web_sm' in your terminal.")

raw_review = "Thiss PHONE is Amazng!!! I buyed it last wek, and the battry life is sooo goood but the camra quality isnt that greaat."
print("Original Text:")
print(raw_review)

Loading spaCy English model...
spaCy model loaded successfully.

Original Text:
Thiss PHONE is Amazng!!! I buyed it last wek, and the battry life is sooo goood but the camra quality isnt that greaat.


### Text Normalization
Before analyzing the text, we must clean and standardize it. This step involves:
*   **Lowercasing:** Converting all characters to lowercase so that words like "PHONE" and "phone" are treated as the same word.
*   **Punctuation Removal:** Stripping out special characters (like `!!!` or `,`) that do not carry semantic meaning for basic text processing.
*   **Whitespace Cleanup:** Removing irregular or trailing spaces to ensure clean string formatting.

In [3]:
norm_text = raw_review.lower()

norm_text = re.sub(f"[{re.escape(string.punctuation)}]", "", norm_text)

norm_text = re.sub(r"\s+", " ", norm_text).strip()

print("--- Output after Normalization ---")
print(norm_text)

--- Output after Normalization ---
thiss phone is amazng i buyed it last wek and the battry life is sooo goood but the camra quality isnt that greaat


### Tokenization
Tokenization is the process of breaking down the continuous normalized string into smaller, individual units called "tokens" (usually words). In this cell, we implement tokenization using two different libraries—**NLTK** and **spaCy**—to prepare the text for word-level analysis.

In [5]:
nltk_tokens = word_tokenize(norm_text)

spacy_doc = nlp(norm_text)
spacy_tokens = [token.text for token in spacy_doc]

print("--- Output after Tokenization ---")
print("NLTK Tokens: ", nltk_tokens)
print("spaCy Tokens:", spacy_tokens)

--- Output after Tokenization ---
NLTK Tokens:  ['thiss', 'phone', 'is', 'amazng', 'i', 'buyed', 'it', 'last', 'wek', 'and', 'the', 'battry', 'life', 'is', 'sooo', 'goood', 'but', 'the', 'camra', 'quality', 'isnt', 'that', 'greaat']
spaCy Tokens: ['this', 's', 'phone', 'is', 'amazng', 'i', 'buyed', 'it', 'last', 'wek', 'and', 'the', 'battry', 'life', 'is', 'sooo', 'goood', 'but', 'the', 'camra', 'quality', 'is', 'nt', 'that', 'greaat']


### Stop-words Removal
Customer reviews often contain common filler words such as "is", "it", "and", and "the". These are known as stop-words. Because they occur frequently but offer little analytical value, we remove them to isolate the meaningful keywords. We will filter these out using the default stop-word lists from both NLTK and spaCy.

In [6]:
nltk_stop_words = set(stopwords.words('english'))
nltk_filtered = [word for word in nltk_tokens if word not in nltk_stop_words]

spacy_filtered = [token.text for token in spacy_doc if not token.is_stop]

print("--- Output after Stop-word Removal ---")
print("NLTK Filtered: ", nltk_filtered)
print("spaCy Filtered:", spacy_filtered)

--- Output after Stop-word Removal ---
NLTK Filtered:  ['thiss', 'phone', 'amazng', 'buyed', 'last', 'wek', 'battry', 'life', 'sooo', 'goood', 'camra', 'quality', 'isnt', 'greaat']
spaCy Filtered: ['s', 'phone', 'amazng', 'buyed', 'wek', 'battry', 'life', 'sooo', 'goood', 'camra', 'quality', 'nt', 'greaat']


### Stemming and Lemmatization
To further standardize the vocabulary, we reduce words to their base or root forms using two techniques:
*   **Stemming (NLTK Porter Stemmer):** A rule-based approach that crudely chops off suffixes (e.g., stripping "ing" or "ly"). It is fast but can result in non-dictionary words.
*   **Lemmatization (NLTK WordNet & spaCy):** A more advanced morphological analysis that maps words back to their actual dictionary root (lemma), considering context and part of speech.

In [7]:
stemmer = PorterStemmer()
nltk_stemmed = [stemmer.stem(word) for word in nltk_filtered]

lemmatizer = WordNetLemmatizer()
nltk_lemmatized = [lemmatizer.lemmatize(word) for word in nltk_filtered]

spacy_lemmatized = [token.lemma_ for token in spacy_doc if not token.is_stop]

print("--- Output after Stemming & Lemmatization ---")
print("NLTK Stemmed:    ", nltk_stemmed)
print("NLTK Lemmatized: ", nltk_lemmatized)
print("spaCy Lemmatized:", spacy_lemmatized)

--- Output after Stemming & Lemmatization ---
NLTK Stemmed:     ['thiss', 'phone', 'amazng', 'buy', 'last', 'wek', 'battri', 'life', 'sooo', 'goood', 'camra', 'qualiti', 'isnt', 'greaat']
NLTK Lemmatized:  ['thiss', 'phone', 'amazng', 'buyed', 'last', 'wek', 'battry', 'life', 'sooo', 'goood', 'camra', 'quality', 'isnt', 'greaat']
spaCy Lemmatized: ['s', 'phone', 'amazng', 'buy', 'wek', 'battry', 'life', 'sooo', 'goood', 'camra', 'quality', 'not', 'greaat']


### Spelling Correction via Edit Distance
The raw review contains spelling errors like "battry" and "wek". To quantify these errors, we use **Edit Distance (Levenshtein distance)**. This algorithm calculates the minimum number of single-character operations—insertions, deletions, or substitutions—required to transform a misspelled word into its correct dictionary form. We will compute this using substitution costs of 1 and 2 to observe how penalty weights affect the calculation.

In [8]:
misspelled_word = "battry"
correct_word = "battery"

ed_cost_1 = edit_distance(misspelled_word, correct_word, substitution_cost=1)
ed_cost_2 = edit_distance(misspelled_word, correct_word, substitution_cost=2)

print("--- Output for Edit Distance ---")
print(f"Target Correction: '{misspelled_word}' -> '{correct_word}'")
print(f"Edit Distance (Substitution Cost = 1): {ed_cost_1}")
print(f"Edit Distance (Substitution Cost = 2): {ed_cost_2}")

--- Output for Edit Distance ---
Target Correction: 'battry' -> 'battery'
Edit Distance (Substitution Cost = 1): 1
Edit Distance (Substitution Cost = 2): 1


### Conclusion and Output Analysis

In this lab assignment, I successfully built a text-preprocessing pipeline tailored for noisy e-commerce reviews. By comparing NLTK and spaCy, we observed distinct differences in how different NLP libraries handle informal, misspelled text.

**Summary of Observations based on Generated Outputs:**

1. **Normalization:** The raw string was successfully cleaned, stripping out punctuation and converting everything to lowercase:
   * `"thiss phone is amazng i buyed it last wek and the battry life is sooo goood but the camra quality isnt that greaat"`
2. **Tokenization & Stop-word Removal (NLTK vs. spaCy):**
   * **NLTK** treated misspelled or combined words like `"thiss"` and `"isnt"` as single tokens.
   * **spaCy** applies a more complex linguistic model, splitting `"thiss"` into `['this', 's']` and `"isnt"` into `['is', 'nt']`. Consequently, during stop-word removal, spaCy filtered out `"this"` and `"is"`, leaving behind `"s"` and `"nt"`. spaCy also removed `"last"`, which NLTK retained.
3. **Stemming vs. Lemmatization:**
   * The **NLTK Porter Stemmer** aggressively truncated word endings, reducing `"buyed"` to `"buy"`, but also creating non-dictionary words like `"battri"` and `"qualiti"`.
   * The **NLTK Lemmatizer** struggled with the grammatical errors (leaving `"buyed"` as is), whereas the **spaCy Lemmatizer** successfully mapped `"buyed"` back to its dictionary root `"buy"`, and expanded `"nt"` to `"not"`.
4. **Edit Distance:**
   * When correcting `"battry"` to `"battery"`, the edit distance evaluated to **1**. Because this correction only requires a single insertion (the letter 'e'), no substitutions were made. Therefore, changing the substitution cost from 1 to 2 had no impact on the final score.

**Final Takeaway:**
Real-world text data is highly unstructured. This exercise highlights that there is no "one-size-fits-all" library. Rule-based approaches (like NLTK's stemmer) are fast but rigid, while model-based approaches (like spaCy) attempt deeper linguistic understanding but can behave unpredictably with heavy spelling errors. Proper preprocessing and spelling correction (via Edit Distance) are essential steps before passing this data into advanced machine learning models.